In [1]:
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch
print("cuda available:", torch.cuda.is_available())
print("imported torch")
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
print("imported tokeniser")
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from datasets import Dataset
print("cuda" if torch.cuda.is_available() else "cpu")
import utils


cuda available: True
imported torch
imported tokeniser
cuda


In [2]:
RANDOM_SEED = 47

train_df, dev_df, test_df = utils.load_and_clean_data()

# train_df, val_df = train_test_split(
#     train_df,
#     test_size=0.15,
#     stratify=train_df["binary_label"],  # ensures same class ratio in both splits
#     random_state=RANDOM_SEED
# )

# majority = train_df[train_df["binary_label"] == 0]
# minority = train_df[train_df["binary_label"] == 1]

# # Upsample minority to match majority
# minority_upsampled = resample(
#     minority,
#     replace=True,                    # sample with replacement
#     n_samples=len(majority) // 2,         # match majority class size
#     random_state=RANDOM_SEED
# )

# train_balanced = pd.concat([majority, minority_upsampled]).sample(
#     frac=1, random_state=RANDOM_SEED
# ).reset_index(drop=True)

# print(f"Original: {len(train_df)} | Balanced: {len(train_balanced)}")
# print(train_balanced["binary_label"].value_counts())

Data loaded successfully
Train: 8375 | Dev: 2094 | Test: 3831
train: 0 NaN | 0 empty | 1 <3 words
Short rows (<3 words):      par_id      text  binary_label
1620   1657  refugees             0
dev: 1 NaN | 0 empty | 1 <3 words
Short rows (<3 words):     par_id              text  binary_label
794   9064  Feeling hopeless             0
  Dropping 1 rows:
    par_id text
401   8640  NaN
test: 0 NaN | 0 empty | 0 <3 words


In [3]:
MODEL_NAME = "roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(df):
    enc = tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    if "binary_label" in df.columns:
        enc["labels"] = df["binary_label"].tolist()
    return Dataset.from_dict(enc)

train_dataset = tokenize(train_df)
# val_dataset   = tokenize(val_df)
dev_dataset   = tokenize(dev_df)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
labels_array = train_df["binary_label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=labels_array
)
print(f"Class weights → 0: {class_weights[0]:.3f}, 1: {class_weights[1]:.3f}")

# Move to device for use in loss
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
device = torch.device(device)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

Class weights → 0: 0.552, 1: 5.274
Device: cuda


In [5]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.dtype))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss
    
from transformers import TrainerCallback

class PrinterCallback(TrainerCallback):
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}", flush=True)


In [6]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # F1 of positive class only — matches the task metric
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_pcl": f1}



In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir                  = "./model",
    num_train_epochs            = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_pcl",
    bf16                        = False,
    fp16                        = False,
    logging_steps               = 1,
    # report_to                 = "none",
    # disable_tqdm              = True,
)

trainer = Trainer( # WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [PrinterCallback()],
)

trainer.train()


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Pcl
1,0.038232,0.232688,0.548000
2,0.010468,0.211046,0.573427
3,0.006747,0.268015,0.530351
4,0.061170,0.318152,0.583756


Step 1: {'loss': 0.6750368475914001, 'grad_norm': 5.42714262008667, 'learning_rate': 0.0, 'epoch': 0.0019083969465648854}
Step 2: {'loss': 0.6867613792419434, 'grad_norm': 6.034207344055176, 'learning_rate': 4.7619047619047627e-08, 'epoch': 0.003816793893129771}
Step 3: {'loss': 0.6855542659759521, 'grad_norm': 5.709794998168945, 'learning_rate': 9.523809523809525e-08, 'epoch': 0.0057251908396946565}
Step 4: {'loss': 0.6997002363204956, 'grad_norm': 4.546203136444092, 'learning_rate': 1.4285714285714287e-07, 'epoch': 0.007633587786259542}
Step 5: {'loss': 0.676587700843811, 'grad_norm': 5.833095550537109, 'learning_rate': 1.904761904761905e-07, 'epoch': 0.009541984732824428}
Step 6: {'loss': 0.6713649034500122, 'grad_norm': 4.595951557159424, 'learning_rate': 2.3809523809523811e-07, 'epoch': 0.011450381679389313}
Step 7: {'loss': 0.6916139125823975, 'grad_norm': 6.526584148406982, 'learning_rate': 2.8571428571428575e-07, 'epoch': 0.013358778625954198}
Step 8: {'loss': 0.669198095798492

...

Step 1573: {'loss': 0.0034999470226466656, 'grad_norm': 0.20753805339336395, 'learning_rate': 2.7783669141039237e-06, 'epoch': 3.0019083969465647}
Step 1574: {'loss': 0.3808993101119995, 'grad_norm': 16.795757293701172, 'learning_rate': 2.7730646871686113e-06, 'epoch': 3.00381679389313}
Step 1575: {'loss': 0.013701407238841057, 'grad_norm': 2.4049553871154785, 'learning_rate': 2.767762460233298e-06, 'epoch': 3.0057251908396947}
Step 1576: {'loss': 0.533826470375061, 'grad_norm': 30.526851654052734, 'learning_rate': 2.7624602332979856e-06, 'epoch': 3.0076335877862594}
Step 1577: {'loss': 0.003534178249537945, 'grad_norm': 0.24945342540740967, 'learning_rate': 2.7571580063626724e-06, 'epoch': 3.0095419847328246}
Step 1578: {'loss': 0.2986912727355957, 'grad_norm': 27.453195571899414, 'learning_rate': 2.75185577942736e-06, 'epoch': 3.0114503816793894}
Step 1579: {'loss': 0.0054979948326945305, 'grad_norm': 0.568846583366394, 'learning_rate': 2.7465535524920467e-06, 'epoch': 3.013358778625

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Step 2096: {'train_runtime': 517.4538, 'train_samples_per_second': 64.74, 'train_steps_per_second': 4.051, 'total_flos': 4407110177280000.0, 'train_loss': 0.19488308812937707, 'epoch': 4.0}


TrainOutput(global_step=2096, training_loss=0.19488308812937707, metrics={'train_runtime': 517.4538, 'train_samples_per_second': 64.74, 'train_steps_per_second': 4.051, 'total_flos': 4407110177280000.0, 'train_loss': 0.19488308812937707, 'epoch': 4.0})

In [11]:
def get_predictions(trainer, dataset):
    preds_output = trainer.predict(dataset)
    logits = preds_output.predictions
    return np.argmax(logits, axis=-1)

# Dev predictions
dev_preds = get_predictions(trainer, dev_dataset)
print(classification_report(dev_df["binary_label"].values, dev_preds, target_names=["No PCL", "PCL"]))
np.savetxt("dev.txt", dev_preds.astype(int), fmt="%d")

# Test predictions (no labels)
test_dataset = tokenize(test_df)
test_preds = get_predictions(trainer, test_dataset)
np.savetxt("test.txt", test_preds.astype(int), fmt="%d")


              precision    recall  f1-score   support

      No PCL       0.96      0.96      0.96      1894
         PCL       0.59      0.58      0.58       199

    accuracy                           0.92      2093
   macro avg       0.77      0.77      0.77      2093
weighted avg       0.92      0.92      0.92      2093

